# VECTRI Core Equations Demo (Single Location)

This notebook computes key VECTRI quantities for a single grid cell:
- Larval development and survival
- Gonotrophic and sporogonic cycles
- Adult vector survival
- Human biting rate, daily EIR, and infection probabilities
- A simple rainfall-driven pond hydrology state

It uses synthetic climate data (temperature and rainfall) but can be adapted to real data.


In [ ]:
from dataclasses import dataclass
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
@dataclass
class VectriParams:
    # Larval development
    T_L_min: float = 16.0     # min water temp for larval dev (C)
    K_L: float = 90.9         # larval degree-days
    T_L_max: float = 37.0     # lethal upper water temp (C)

    # Larval survival
    P_L_surv0: float = 0.825  # base daily larval survival (no crowding/flush)
    M_L_max: float = 300.0    # larval biomass capacity (mg m^-2)
    tau_flush: float = 50.0   # rainfall scale for flushing (mm/day)
    K_flush_inf: float = 0.4  # asymptotic survival of 1st-stage larvae in extreme rain

    # Gonotrophic cycle
    T_gono_min: float = 7.7   # min temp for egg dev (C)
    K_gono: float = 37.1      # gonotrophic degree-days

    # Sporogonic (parasite) cycle
    T_sporo_min: float = 16.0 # min temp for parasite dev (C)
    K_sporo: float = 111.0    # sporogonic degree-days

    # Adult survival: Martens II
    K_mar2_0: float = -4.4
    K_mar2_1: float = 1.31
    K_mar2_2: float = -0.03

    # Indoor temperature parameterization
    T0_indoor: float = 10.33
    K_indoor: float = 0.58

    # Host-vector system
    tau_zoo: float = 50.0     # human density scale for zoophily (people/km^2)
    P_hv: float = 0.2         # P(mosquito infected | bite on infectious host)
    P_vh: float = 0.3         # P(host infected | infectious bite)

@dataclass
class HydroParams:
    w_max: float = 0.04   # max fractional pond coverage (4% of grid cell)
    E: float = 5.0        # evaporation (mm/day)
    I: float = 245.0      # infiltration (mm/day)
    K_w: float = 0.001    # geometry/scale factor for converting water balance to area

params = VectriParams()
hydro = HydroParams()


In [ ]:
def water_temperature(T2m: float, delta_T: float = 1.5) -> float:
    """Simple offset: shallow water slightly warmer than air."""
    return T2m + delta_T


def larval_development_rate(Twat: float, params=params) -> float:
    """
    Degree-day larval development rate (fraction of lifecycle per day).
    Returns 0 if outside viable temperature range.
    """
    if Twat <= params.T_L_min or Twat >= params.T_L_max:
        return 0.0
    return (Twat - params.T_L_min) / params.K_L


def flushing_factor(Rd: float, Lf: float, params=params) -> float:
    """
    Rainfall-driven larval flushing factor K_flush (0-1).
    Lf is larval fractional stage in [0,1] (0=early, 1=late).
    """
    Lf = max(0.0, min(1.0, Lf))
    inner = ((1 - params.K_flush_inf)
             * math.exp(-Rd / params.tau_flush)
             + params.K_flush_inf)
    return Lf + (1 - Lf) * inner


def larval_survival(M_L: float, w: float, K_flush: float, params=params) -> float:
    """
    Daily larval survival probability including crowding and flushing.
    M_L: total larval biomass (mg m^-2)
    w: fractional breeding area (0-1)
    """
    if w <= 0.0:
        return 0.0  # no ponds, no larvae

    # Crowding term, capped in [0,1]
    crowd_term = 1.0 - M_L / (w * params.M_L_max)
    crowd_term = max(0.0, min(1.0, crowd_term))

    P_surv = crowd_term * K_flush * params.P_L_surv0
    return max(0.0, min(1.0, P_surv))


In [ ]:
def gonotrophic_rate(T_eff: float, params=params):
    """
    Temperature-dependent gonotrophic development rate and period.
    Returns (rate per day, period in days).
    """
    if T_eff <= params.T_gono_min:
        return 0.0, math.inf
    R = (T_eff - params.T_gono_min) / params.K_gono
    period_days = 1.0 / R
    return R, period_days


def sporogonic_rate(T_eff: float, params=params):
    """
    Temperature-dependent parasite development rate and EIP.
    Returns (rate per day, EIP in days).
    """
    if T_eff <= params.T_sporo_min:
        return 0.0, math.inf
    R = (T_eff - params.T_sporo_min) / params.K_sporo
    EIP_days = 1.0 / R
    return R, EIP_days


def vector_survival_prob(T_eff: float, params=params) -> float:
    """
    Adult mosquito daily survival probability (Martens II form).
    """
    den = params.K_mar2_0 + params.K_mar2_1 * T_eff + params.K_mar2_2 * T_eff**2
    if den <= 0:
        return 0.0
    P = math.exp(-1.0 / den)
    return max(0.0, min(1.0, P))


In [ ]:
def indoor_temperature(T2m: float, params=params) -> float:
    """Indoor temperature parameterization."""
    return params.T0_indoor + params.K_indoor * T2m


def effective_temperature(T2m: float, beta_indoor: float = 0.0, params=params) -> float:
    """
    Weighted temperature seen by mosquitoes, based on fraction of time indoors.
    beta_indoor = 0 -> always outdoors; 1 -> always indoors.
    """
    T_ind = indoor_temperature(T2m, params)
    return beta_indoor * T_ind + (1 - beta_indoor) * T2m


def human_biting_rate(V_biting: float, H: float, params=params) -> float:
    """
    Mean human biting rate (bites per person per day).
    V_biting: number of biting mosquitoes in the cell
    H: human population in the cell
    """
    if H <= 0:
        return 0.0
    phi = 1.0 - math.exp(-H / params.tau_zoo)  # zoophily factor
    return phi * V_biting / H


def daily_eir(hbr_bar: float, CSPR: float) -> float:
    """Daily EIR = mean biting rate times circumsporozoite protein rate."""
    return hbr_bar * CSPR


In [ ]:
def prob_host_to_vector(H_inf: float, H: float, params=params) -> float:
    """
    Probability a blood meal infects the mosquito.
    H_inf: number of infectious humans
    H: total humans
    """
    if H <= 0:
        return 0.0
    return (H_inf / H) * params.P_hv


def prob_vector_to_host(EIR_d: float, params=params) -> float:
    """
    Daily infection probability for a susceptible host.
    Uses Poisson number of infectious bites with mean EIR_d.
    Closed form: 1 - exp(-lambda * p).
    """
    return 1.0 - math.exp(-EIR_d * params.P_vh)


In [ ]:
def update_pond_fraction(w_prev: float, rain: float, hydro=hydro, dt: float = 1.0) -> float:
    """
    Simple pond hydrology:
      dw/dt = K_w * [ rain * (w_max - w) - w * (E + I) ]
    """
    dw = hydro.K_w * (rain * (hydro.w_max - w_prev) - w_prev * (hydro.E + hydro.I))
    w_new = w_prev + dw * dt
    # keep within [0, w_max]
    w_new = max(0.0, min(hydro.w_max, w_new))
    return w_new


## Time Series Extension (Single Location)

We now:
- Build a daily synthetic time series of temperature and rainfall.
- Evolve a simple pond fraction state w(t) using a toy hydrology model.
- Compute, for each day:
  - Temperatures (air, water, effective)
  - Larval development and survival
  - Gonotrophic and sporogonic rates, EIP
  - Adult survival probability
  - Human biting rate, daily EIR, and infection probabilities


In [ ]:
# --- Time axis: e.g. 180 days ---
dates = pd.date_range("2025-01-01", periods=180, freq="D")
day_of_year = dates.dayofyear.values

# --- Synthetic temperature (C): weak seasonal cycle around 24C ---
T2m_series = 24.0 + 3.0 * np.sin(2 * np.pi * (day_of_year / 365.0))

# --- Synthetic rainfall (mm/day): random-ish daily rain ---
rng = np.random.default_rng(42)
rain_series = rng.gamma(shape=2.0, scale=3.0, size=len(dates))  # avg ~6 mm/day

climate_df = pd.DataFrame(
    {
        "date": dates,
        "T2m": T2m_series,
        "rain": rain_series,
    }
).set_index("date")

climate_df.head()


### Location-specific constants

In [ ]:
# Hydrology / larvae parameters
M_L_const = 3.0         # larval biomass (mg m^-2)
L_f_const = 0.25        # early larvae
delta_T_water = 1.5     # water - air temp offset (C)
w_init = 0.0            # initial pond fraction (completely dry)

# Host / vector parameters
H_const = 200.0         # humans in the cell
H_inf_frac_const = 0.10 # 10% infectious humans
V_biting_const = 50.0   # biting mosquitoes
CSPR_const = 0.10       # 10% of mosquitoes infectious
beta_indoor_const = 0.5 # 50% of time indoors


### Loop over days and compute all quantities

In [ ]:
results = []

w = w_init  # pond fraction at start

for date, row in climate_df.iterrows():
    T2m = row["T2m"]
    rain = row["rain"]

    # Hydrology: update pond fraction
    w = update_pond_fraction(w, rain, hydro)

    # Temperatures
    Twat = water_temperature(T2m, delta_T_water)
    T_eff = effective_temperature(T2m, beta_indoor_const)

    # Larval development & survival
    R_L = larval_development_rate(Twat)
    larval_period = math.inf if R_L == 0 else 1.0 / R_L

    K_fl = flushing_factor(rain, L_f_const)
    P_L_surv = larval_survival(M_L_const, w, K_fl)

    # Adult + parasite
    R_gono, gono_period = gonotrophic_rate(T_eff)
    R_sporo, EIP_days = sporogonic_rate(T_eff)
    P_V_surv = vector_survival_prob(T_eff)

    # Host community
    H = H_const
    H_inf = H * H_inf_frac_const
    hbr_bar = human_biting_rate(V_biting_const, H)
    EIR_d = daily_eir(hbr_bar, CSPR_const)

    P_h2v = prob_host_to_vector(H_inf, H)
    P_v2h = prob_vector_to_host(EIR_d)

    results.append(
        {
            "date": date,
            "T2m": T2m,
            "rain": rain,
            "w": w,
            "Twat": Twat,
            "T_eff": T_eff,
            "R_L": R_L,
            "larval_period_days": larval_period,
            "K_flush": K_fl,
            "P_L_surv": P_L_surv,
            "R_gono": R_gono,
            "gono_period_days": gono_period,
            "R_sporo": R_sporo,
            "EIP_days": EIP_days,
            "P_V_surv": P_V_surv,
            "hbr": hbr_bar,
            "EIR_d": EIR_d,
            "P_h2v": P_h2v,
            "P_v2h": P_v2h,
        }
    )

ts_df = pd.DataFrame(results).set_index("date")
ts_df.head()


### Basic summaries over the period

In [ ]:
print("=== Basic summaries over the period ===")
print(f"Mean T2m           : {ts_df['T2m'].mean():.2f} C")
print(f"Mean Twat          : {ts_df['Twat'].mean():.2f} C")
print(f"Mean larval R_L    : {ts_df['R_L'].mean():.4f} frac/day")
print(f"Mean larval survival: {ts_df['P_L_surv'].mean():.3f}")
print(f"Mean EIP           : {ts_df['EIP_days'].replace(math.inf, float('nan')).mean():.2f} days")
print(f"Mean P_V_surv      : {ts_df['P_V_surv'].mean():.3f}")
print(f"Mean hbr           : {ts_df['hbr'].mean():.3f} bites/person/day")
print(f"Mean daily EIR     : {ts_df['EIR_d'].mean():.4f} inf. bites/person/day")
print(f"Mean P_v2h (per day): {ts_df['P_v2h'].mean():.4f}")


## 4. Plots

### 4.1 Air vs water temperature


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts_df.index, ts_df["T2m"], label="T2m (air)")
plt.plot(ts_df.index, ts_df["Twat"], label="Twat (water)")
plt.ylabel("Temperature (C)")
plt.xlabel("Date")
plt.title("Air vs Water Temperature")
plt.legend()
plt.tight_layout()
plt.show()


### 4.2 Daily rainfall

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(ts_df.index, ts_df["rain"])
plt.ylabel("Rainfall (mm/day)")
plt.xlabel("Date")
plt.title("Daily Rainfall")
plt.tight_layout()
plt.show()


### 4.3 Dynamic pond coverage (w)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts_df.index, ts_df["w"])
plt.ylabel("Pond fraction w")
plt.xlabel("Date")
plt.title("Dynamic Pond Coverage")
plt.tight_layout()
plt.show()


### 4.4 Daily larval survival probability

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts_df.index, ts_df["P_L_surv"])
plt.ylabel("Larval survival (daily)")
plt.xlabel("Date")
plt.title("Daily Larval Survival Probability")
plt.tight_layout()
plt.show()


### 4.5 Gonotrophic period and EIP

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts_df.index, ts_df["gono_period_days"], label="Gonotrophic period (days)")
plt.plot(ts_df.index, ts_df["EIP_days"], label="EIP (sporogonic, days)")
plt.ylabel("Days")
plt.xlabel("Date")
plt.title("Gonotrophic Period and EIP")
plt.legend()
plt.tight_layout()
plt.show()


### 4.6 Daily EIR (infectious bites per person per day)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts_df.index, ts_df["EIR_d"])
plt.ylabel("Daily EIR\n(infectious bites/person/day)")
plt.xlabel("Date")
plt.title("Daily Entomological Inoculation Rate (EIR)")
plt.tight_layout()
plt.show()
